In [1]:
import sys
import importlib
sys.path.append('../')  # Adjust the path as needed

import utilities.functions as functions

# Reload the module to reflect the changes
importlib.reload(functions)

<module 'utilities.functions' from '/Users/xuechenkan/potts_model_test/ms1/../utilities/functions.py'>

In [2]:
IN_syn_pairs = ['G140S-Q148H', 'Y143C-S230R']
PR_syn_pairs = ['D30N-N88D', 'V32I-I47V']
RT_syn_pairs = ['K101E-G190S', 'K103N-P225H']

IN_seq_path = 'IN/data/in.reduce4.seq'
PR_seq_path = 'PR/data/pr.exper.reduce4.seq'    
RT_seq_path = 'RT/data/rt.reduce4.seq'

IN_all_seq = functions.read_seq('IN/data/in.reduce4.seq')
PR_all_seq = functions.read_seq('PR/data/pr.exper.reduce4.seq')
RT_all_seq = functions.read_seq('RT/data/rt.reduce4.seq')

IN_consensus = 'IN/data/in.consensus.reduce4.seq'
PR_consensus = 'PR/data/pr.consensus.reduce4.seq'
RT_consensus = 'RT/data/rt.consensus.reduce4.seq'

IN_redux = functions.get_redu_dict('IN/data/in.reduce4.redux',1)
PR_redux = functions.get_redu_dict('PR/data/pr.reduce4.redux',0)
RT_redux = functions.get_redu_dict('RT/data/rt.reduce4.redux',0)

IN_J = functions.load_J_dict('IN/data/J.npy',1,263)
PR_J = functions.load_J_dict('PR/data/J_PR.npy',1,99)
RT_J = functions.load_J_dict('RT/data/J_RT.npy',39,226)

IN_all_seq_unreduced = functions.read_seq('IN/data/in.fullseq')
PR_all_seq_unreduced = functions.read_seq('PR/data/pr.exper.fullseq')
RT_all_seq_unreduced = functions.read_seq('RT/data/rt.fullseq')


# Convert synergistic pairs to reduced format
IN_syn_pairs_reduced = functions.pairs_to_reduced(IN_redux, IN_syn_pairs)
PR_syn_pairs_reduced = functions.pairs_to_reduced(PR_redux, PR_syn_pairs)
RT_syn_pairs_reduced = functions.pairs_to_reduced(RT_redux, RT_syn_pairs)

print("IN reduced pairs:", IN_syn_pairs_reduced)
print("PR reduced pairs:", PR_syn_pairs_reduced)
print("RT reduced pairs:", RT_syn_pairs_reduced)

IN reduced pairs: ['C140D-D148B', 'D143A-D230C']
PR reduced pairs: ['B30D-B88C', 'C32D-A47B']
RT reduced pairs: ['C101A-D190C', 'C103B-D225A']


In [3]:
IN_seq_mut_df = functions.analyze_sequences_mutations(IN_consensus, IN_seq_path)
IN_seq_mut_df.head()

,Sequence,Mutations,Mutations_count
0,ABAAABABACBBAACBDBABBDBDBBBAAAAAABAAABDAABAABA...,"[C7A, A11B, C31A, C50B, A72D, A101B, C124A, B1...",13
1,ABAAABCBACBBAACBDBABCDBDABBAAACAABAAABDAABAABA...,"[A11B, B21C, B25A, D119A, C122B, D125A, D148C,...",13
2,ABAAABCBACBBAACBDBABCDBDABBAAACAABAAABDAABAABA...,"[A11B, B21C, B25A, D119A, C122B, D125A, C140D,...",13
3,ABAAABCBACABAACBABABBDBDBBBBAACAABAAABAAABAABA...,"[D17A, A28B, D39A, D119C, C122B, C124A, D125A,...",9
4,ABAAABCBABABAACBDBABBDBDBBBAAACAABAAABDAABAABA...,"[C10B, A101B, B106A, A155C, C156A, C165A, C195...",9


In [4]:
import pandas as pd

# Load J matrix and create J_dict for IN
IN_J_file = 'IN/data/in.reduce4.J'
IN_max_position = len(IN_all_seq[0])

# Get consensus sequence
with open(IN_consensus, 'r') as f:
    IN_consensus_seq = f.read().strip()

# Calculate energies for each reduced pair
IN_results = []

for pair in IN_syn_pairs_reduced:
    mut1, mut2 = pair.split('-')
    de1_consensus = functions.calculate_delta_e(mut1, IN_consensus_seq, IN_J, 1, 263)
    de2_consensus = functions.calculate_delta_e(mut2, IN_consensus_seq, IN_J, 1, 263)
    de12_consensus = functions.calculate_delta_e_double(mut1, mut2, IN_consensus_seq, IN_J, 1, 263)

    for idx, row in IN_seq_mut_df.iterrows():
        seq_idx = idx
        seq = row['Sequence']
        unreduced_seq = IN_all_seq_unreduced[seq_idx]
        mutations = row['Mutations']
        # mutations_unreduced = functions.reduced_to_unreduced_list(IN_redux, mutations, IN_all_seq_unreduced)
        de1 = functions.calculate_delta_e(mut1, seq, IN_J, 1, 263)
        de2 = functions.calculate_delta_e(mut2, seq, IN_J, 1, 263)
        de12 = functions.calculate_delta_e_double(mut1, mut2, seq, IN_J, 1, 263)
        
        if de12 is None:
            # print(f"Warning: Could not calculate de12 for row {idx}. Skipping.")
            continue
        dde = de12 - de1 - de2
        
        # FLIPPPPPPPPPPP
        if (de1_consensus-de2_consensus) * (de1-de2) < 0 :
            IN_results.append({
                'Epistatis_type': 'FLIP',
                'sequence': unreduced_seq,
                'original_pair': IN_syn_pairs[IN_syn_pairs_reduced.index(pair)],
                'mutations': mutations,
                'de1': de1,
                'de2': de2,
                'de12': de12,
                'dde': dde

            })
        
        elif (de12 > de1 or de12 > de2) and not (de12 > de1 and de12 > de2):
            IN_results.append({
                'Epistatis_type': 'COMPENSATORY',
                'sequence': unreduced_seq,
                'original_pair': IN_syn_pairs[IN_syn_pairs_reduced.index(pair)],
                'mutations': mutations,
                'de1': de1,
                'de2': de2,
                'de12': de12,
                'dde': dde
            })
        
        elif de12 > de1 and de12 > de2:
            IN_results.append({
                'Epistatis_type': 'RESCUE',
                'sequence': unreduced_seq,
                'original_pair': IN_syn_pairs[IN_syn_pairs_reduced.index(pair)],
                'mutations': mutations,
                'de1': de1,
                'de2': de2,
                'de12': de12,
                'dde': dde
            })
        else:
            IN_results.append({
                'Epistatis_type': 'OTHER',
                'sequence': unreduced_seq,
                'original_pair': IN_syn_pairs[IN_syn_pairs_reduced.index(pair)],
                'mutations': mutations,
                'de1': de1,
                'de2': de2,
                'de12': de12,
                'dde': dde
            })
        
        # Group results by original pair and epistatic type, keeping only first 2 of each type
        pair_type_counts = {}
        filtered_indices = []


# Display results
IN_energy_df = pd.DataFrame(IN_results)
print(IN_energy_df)

     Epistatis_type                                           sequence  \
0              FLIP  FLDGIDKAQDEHEKYHSNWRAMASDFNLPPVVAKEIVASCDKCQLK...   
1      COMPENSATORY  FLDGIDKAQEEHEKYHSNWRAMASDFNLPPVIAKEIVACCDKCQLK...   
2      COMPENSATORY  FLDGIDKAQEDHEKYHSNWRAMASDFNLPPMIAKEIVACCDKCQLK...   
3            RESCUE  FLDGIDKAQEEHEKYHSNWKAMVSDFNLPPVVAKEIVASCDKCQLK...   
4            RESCUE  FLDGIDKAQEDHEKYHSNWRTMVSEFNLPPVVAKEIVASCDKCQLK...   
...             ...                                                ...   
1904           FLIP  FLDGIDKAQEEHEKYHNNWRAMASDFNIPPVVAKEIVASCDKCQIK...   
1905          OTHER  FLDGIDKAQEEHEKYHSNWRAMASDFNLPPVVAKEIVASCDKCQIK...   
1906   COMPENSATORY  FLDGIDKAQEEHEKYHNNWRAMASDFNLPPIVAKEIVASCDKCQLK...   
1907           FLIP  FLDGIDKAQEEHEKYHSNWRAMASDFNLPPIVAKEIVASCDKCQLK...   
1908           FLIP  FLDGIDKAQEDHEKYHSNWRAMASDFNIPPIIAKEIIASCDKCQLK...   

     original_pair                                          mutations  \
0      G140S-Q148H  [C10B, A101B, B106

In [5]:
# import csv

# IN_reduced_lists = [
#     ['C10B', 'A101B', 'B106A', 'A155C', 'C156A', 'C165A', 'C195B', 'B201A', 'C220D'],
#     ['A11B', 'C31A', 'A101B', 'B111C', 'D119C', 'C232A', 'B256A'],
#     ['A32B', 'D39A', 'C50A', 'D112B', 'C113D', 'C124D', 'A155C', 'A163D', 'B201A', 'C234A', 'D255A'],
#     ['A11B', 'C31B', 'A32B', 'D39A', 'A72D', 'A92C', 'A101B', 'B111D', 'D119C', 'B135C', 'A155C', 'A193C', 'B201A', 'D218C'],
#     ['B20A', 'B23D', 'A72D', 'A92C', 'C124A', 'B204A', 'B206A'],
#     ['A11B', 'B21C', 'B23D', 'B25A', 'B157A', 'B201A'],
#     ['C7A', 'A11B', 'C31A', 'C50B', 'A72D', 'A101B', 'C124A', 'B135C', 'C140D', 'D148B', 'B200A', 'B201A', 'C220B'],
#     ['D17A', 'A28B', 'D39A', 'D119C', 'C122B', 'C124A', 'D125A', 'C140D', 'D148B'],
#     ['A72D', 'C124A', 'C156A', 'A167B'],
#     ['A14B', 'D22C', 'A32B', 'A37B', 'D39A', 'A101B', 'A163D', 'B201A', 'D253A'],
#     ['D17A', 'B20A', 'A28B', 'D39A', 'C124A', 'D125A', 'A155C', 'B201A', 'C208A'],
#     ['C31A', 'A54B', 'C113D', 'C124A', 'D125A', 'B201A', 'C215B']
# ]

# IN_unreduced_lists = []
# for li in IN_reduced_lists:
#     IN_unreduced_lists.append(functions.reduced_to_unreduced_list(IN_redux, li, IN_all_seq_unreduced))
# print(IN_unreduced_lists)
    
# # Write the list to a CSV file
# with open('IN_unreduced_lists.csv', 'w', newline='') as csvfile:
#     writer = csv.writer(csvfile)
#     writer.writerow(IN_unreduced_lists)

In [6]:
PR_seq_mut_df = functions.analyze_sequences_mutations(PR_consensus, PR_seq_path)

# Get consensus sequence
with open(PR_consensus, 'r') as f:
    PR_consensus_seq = f.read().strip()

# Calculate energies for each reduced pair
PR_results = []

for pair in PR_syn_pairs_reduced:
    mut1, mut2 = pair.split('-')
    de1_consensus = functions.calculate_delta_e(mut1, PR_consensus_seq, PR_J, 1, 99)
    de2_consensus = functions.calculate_delta_e(mut2, PR_consensus_seq, PR_J, 1, 99)
    de12_consensus = functions.calculate_delta_e_double(mut1, mut2, PR_consensus_seq, PR_J, 1, 99)

    for idx, row in PR_seq_mut_df.iterrows():
        seq = row['Sequence']
        mutations = row['Mutations']
        de1 = functions.calculate_delta_e(mut1, seq, PR_J, 1, 99)
        de2 = functions.calculate_delta_e(mut2, seq, PR_J, 1, 99)
        de12 = functions.calculate_delta_e_double(mut1, mut2, seq, PR_J, 1, 99)
        
        if de12 is None:
            continue
        dde = de12 - de1 - de2
        
        # FLIP
        if (de1_consensus-de2_consensus) * (de1-de2) < 0:
            PR_results.append({
                'Epistatis_type': 'FLIP',
                'original_pair': PR_syn_pairs[PR_syn_pairs_reduced.index(pair)],
                'mutations': mutations,
                'de1': de1,
                'de2': de2,
                'de12': de12,
                'dde': dde
            })
        
        elif (de12 > de1 or de12 > de2) and not (de12 > de1 and de12 > de2):
            PR_results.append({
                'Epistatis_type': 'COMPENSATORY',
                'original_pair': PR_syn_pairs[PR_syn_pairs_reduced.index(pair)],
                'mutations': mutations,
                'de1': de1,
                'de2': de2,
                'de12': de12,
                'dde': dde
            })
        
        elif de12 > de1 and de12 > de2:
            PR_results.append({
                'Epistatis_type': 'RESCUE',
                'original_pair': PR_syn_pairs[PR_syn_pairs_reduced.index(pair)],
                'mutations': mutations,
                'de1': de1,
                'de2': de2,
                'de12': de12,
                'dde': dde
            })
        else:
            PR_results.append({
                'Epistatis_type': 'OTHER',
                'original_pair': PR_syn_pairs[PR_syn_pairs_reduced.index(pair)],
                'mutations': mutations,
                'de1': de1,
                'de2': de2,
                'de12': de12,
                'dde': dde
            })

# Display results
PR_energy_df = pd.DataFrame(PR_results)
print(PR_energy_df)

      Epistatis_type original_pair  \
0              OTHER     D30N-N88D   
1               FLIP     D30N-N88D   
2               FLIP     D30N-N88D   
3               FLIP     D30N-N88D   
4       COMPENSATORY     D30N-N88D   
...              ...           ...   
10520           FLIP     V32I-I47V   
10521          OTHER     V32I-I47V   
10522          OTHER     V32I-I47V   
10523          OTHER     V32I-I47V   
10524   COMPENSATORY     V32I-I47V   

                                               mutations       de1       de2  \
0      [D10C, C14D, D35C, D36C, C37B, C54D, A63D, B64... -5.241160 -5.643820   
1      [C32D, D35C, C37B, C46D, A47B, A63D, C73B, D77... -7.670992 -6.936328   
2      [D15C, C32D, C37B, C46D, A47B, A63D, C82B, A92... -6.982893 -6.548922   
3             [D10C, D15C, A24D, C54D, A63D, A71B, C82B] -6.005604 -5.761354   
4                                     [D15C, A63D, B64D] -2.745276 -4.643936   
...                                                  ...       

In [7]:
import csv

PR_reduced_lists = [
    ['C32D', 'D35C', 'C37B', 'C46D', 'A47B', 'A63D', 'C73B', 'D77C', 'C90B', 'A93B'],
    ['D15C', 'C32D', 'C37B', 'C46D', 'A47B', 'A63D', 'C82B', 'A92C', 'A93B'],
    ['D15C', 'A63D', 'B64D'],
    ['D36A', 'A63D'],
    ['D13B', 'D15C', 'D35C', 'A63D', 'A74C'],
    ['C62D', 'A63D', 'C67B', 'C69A', 'D77C', 'A93B'],
    ['D10C', 'C14D', 'D35C', 'D36C', 'C37B', 'C54D', 'A63D', 'B64C', 'A71C', 'C82A', 'C90B', 'A93B'],
    ['B41C', 'C54D', 'C62D', 'A63D', 'D77C', 'C82B'],
    ['D10C', 'C37A', 'B41C', 'C46D', 'C54D', 'C62D', 'A63D', 'A71B', 'D77C', 'C82B', 'C90B', 'A93B'],
    ['D10A', 'A24D', 'D35C', 'C37B', 'C46D', 'A63D', 'C82B', 'A92D'],
    ['D13B', 'C14D', 'A33C', 'D36A', 'C46D', 'C62D', 'A63D', 'A71C', 'C73B', 'C90B', 'A93B'],
    ['D10A', 'D13B', 'C19B', 'C46D', 'A63D', 'C73A', 'C90B']
]

PR_unreduced_lists = []
for li in PR_reduced_lists:
    PR_unreduced_lists.append(functions.reduced_to_unreduced_list(PR_redux, li, PR_all_seq_unreduced))
print(PR_unreduced_lists)
    
# Write the list to a CSV file
with open('PR_unreduced_lists.csv', 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(PR_unreduced_lists)

[['V32I', 'E35D', 'N37D', 'M46I', 'I47V', 'L63P', 'G73S', 'V77I', 'L90M', 'I93L'], ['I15V', 'V32I', 'N37D', 'M46I', 'I47V', 'L63P', 'V82A', 'Q92R', 'I93L'], ['I15V', 'L63P', 'I64L'], ['M36I', 'L63P'], ['I13V', 'I15V', 'E35D', 'L63P', 'T74S'], ['I62V', 'L63P', 'C67F', 'H69K', 'V77I', 'I93L'], ['L10I', 'K14R', 'E35D', 'M36V', 'N37D', 'I54V', 'L63P', 'I64V', 'A71T', 'V82T', 'L90M', 'I93L'], ['R41K', 'I54V', 'I62V', 'L63P', 'V77I', 'V82A'], ['L10I', 'N37S', 'R41K', 'M46I', 'I54V', 'I62V', 'L63P', 'A71V', 'V77I', 'V82A', 'L90M', 'I93L'], ['L10V', 'L24I', 'E35D', 'N37D', 'M46I', 'L63P', 'V82A', 'Q92K'], ['I13V', 'K14R', 'L33I', 'M36I', 'M46I', 'I62V', 'L63P', 'A71T', 'G73S', 'L90M', 'I93L'], ['L10V', 'I13V', 'L19I', 'M46I', 'L63P', 'G73C', 'L90M']]


In [8]:
RT_seq_mut_df = functions.analyze_sequences_mutations(RT_consensus, RT_seq_path)

# Get consensus sequence
with open(RT_consensus, 'r') as f:
    RT_consensus_seq = f.read().strip()

# Calculate energies for each reduced pair
RT_results = []

for pair in RT_syn_pairs_reduced:
    mut1, mut2 = pair.split('-')
    de1_consensus = functions.calculate_delta_e(mut1, RT_consensus_seq, RT_J, 39, 226)
    de2_consensus = functions.calculate_delta_e(mut2, RT_consensus_seq, RT_J, 39, 226)
    de12_consensus = functions.calculate_delta_e_double(mut1, mut2, RT_consensus_seq, RT_J, 39, 226)

    for idx, row in RT_seq_mut_df.iterrows():
        seq = row['Sequence']
        mutations = row['Mutations']
        de1 = functions.calculate_delta_e(mut1, seq, RT_J, 39, 226)
        de2 = functions.calculate_delta_e(mut2, seq, RT_J, 39, 226)
        de12 = functions.calculate_delta_e_double(mut1, mut2, seq, RT_J, 39, 226)
        
        if de12 is None:
            continue
        dde = de12 - de1 - de2
        
        # FLIP
        if (de1_consensus-de2_consensus) * (de1-de2) < 0:
            RT_results.append({
                'Epistatis_type': 'FLIP',
                'original_pair': RT_syn_pairs[RT_syn_pairs_reduced.index(pair)],
                'mutations': mutations,
                'de1': de1,
                'de2': de2,
                'de12': de12,
                'dde': dde
            })
        
        elif (de12 > de1 or de12 > de2) and not (de12 > de1 and de12 > de2):
            RT_results.append({
                'Epistatis_type': 'COMPENSATORY',
                'original_pair': RT_syn_pairs[RT_syn_pairs_reduced.index(pair)],
                'mutations': mutations,
                'de1': de1,
                'de2': de2,
                'de12': de12,
                'dde': dde
            })
        
        elif de12 > de1 and de12 > de2:
            RT_results.append({
                'Epistatis_type': 'RESCUE',
                'original_pair': RT_syn_pairs[RT_syn_pairs_reduced.index(pair)],
                'mutations': mutations,
                'de1': de1,
                'de2': de2,
                'de12': de12,
                'dde': dde
            })
        else:
            RT_results.append({
                'Epistatis_type': 'OTHER',
                'original_pair': RT_syn_pairs[RT_syn_pairs_reduced.index(pair)],
                'mutations': mutations,
                'de1': de1,
                'de2': de2,
                'de12': de12,
                'dde': dde
            })

# Display results
RT_energy_df = pd.DataFrame(RT_results)
print(RT_energy_df)

      Epistatis_type original_pair  \
0              OTHER   K101E-G190S   
1               FLIP   K101E-G190S   
2               FLIP   K101E-G190S   
3               FLIP   K101E-G190S   
4               FLIP   K101E-G190S   
...              ...           ...   
25397   COMPENSATORY   K103N-P225H   
25398   COMPENSATORY   K103N-P225H   
25399   COMPENSATORY   K103N-P225H   
25400   COMPENSATORY   K103N-P225H   
25401   COMPENSATORY   K103N-P225H   

                                               mutations       de1       de2  \
0             [D10C, D45B, C65B, C66A, A84C, B85C, A97C] -5.286482 -5.422381   
1                [D1A, C65B, C135B, B136D, D169B, D173C] -5.306311 -5.231634   
2            [D1B, D10C, D36B, D45B, B85C, B139A, D162A] -3.931431 -3.129288   
3      [D11A, D22B, A29D, C32D, C65B, B85C, A97C, B12... -4.946983 -4.570993   
4                       [D45B, D52C, C65B, D173C, D177C] -5.044549 -4.964806   
...                                                  ...       

In [11]:
import csv

RT_reduced_lists = [['D39A', 'C103B', 'C173B', 'B174D', 'D207B', 'D211C'],
['D39B', 'D48C', 'D74B', 'D83B', 'B123C', 'B177A', 'D200A'],
['D41C', 'C43D', 'A67D', 'B69A', 'C70D', 'C104D', 'B184C', 'B202C', 'D211A', 'D215B', 'C219D'],
['D49A', 'A122C', 'C138B', 'D166A', 'D197C', 'D203C', 'D211A'],
['D41C', 'D74B', 'B98D', 'B108C', 'A122D', 'D166B', 'B177A', 'D181A', 'D200A', 'D207B', 'D215C', 'C221D'],
['D74C', 'D106C', 'A135C', 'D181A', 'C214D', 'C219B'],
['C43D', 'B123C', 'A135C', 'B177A', 'D197A', 'A210D', 'D211C', 'D215C'],
['D41C', 'D74B', 'B98D', 'B108C', 'A122D', 'D166B', 'B177A', 'D181A', 'D200A', 'D207B', 'D215C', 'C221D'],
['B98D', 'C101B', 'B123C', 'A135B', 'B177A', 'B184C', 'D194B', 'D196A', 'D200A'],
['A46C', 'D48C', 'B62D', 'B65C', 'D68B', 'C70B', 'D74C', 'B98D', 'C101B', 'A122C', 'A135C', 'B162A', 'A178D', 'B184C', 'D189A', 'D190C', 'D200A', 'D211C']
]
RT_unreduced_lists = []
for li in RT_reduced_lists:
    RT_unreduced_lists.append(functions.reduced_to_unreduced_list(RT_redux, li, RT_all_seq_unreduced, 39))
print(RT_unreduced_lists)
    
# Write the list to a CSV file
with open('RT_unreduced_lists.csv', 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(RT_unreduced_lists)

[['T39S', 'K103N', 'K173E', 'Q174K', 'Q207E', 'R211K'], ['T39A', 'S48T', 'L74V', 'R83K', 'D123E', 'D177E', 'T200A'], ['M41L', 'K43Q', 'D67N', 'T69N', 'K70R', 'K104N', 'M184V', 'I202V', 'R211T', 'T215F', 'K219Q'], ['K49R', 'K122E', 'E138A', 'K166Q', 'Q197P', 'E203D', 'R211T'], ['M41L', 'L74V', 'A98S', 'V108I', 'K122P', 'K166R', 'D177E', 'Y181C', 'T200A', 'Q207E', 'T215Y', 'H221Y'], ['L74I', 'V106I', 'I135T', 'Y181C', 'F214L', 'K219N'], ['K43Q', 'D123E', 'I135T', 'D177E', 'Q197K', 'L210W', 'R211K', 'T215Y'], ['M41L', 'L74V', 'A98S', 'V108I', 'K122P', 'K166R', 'D177E', 'Y181C', 'T200A', 'Q207E', 'T215Y', 'H221Y'], ['A98S', 'K101Q', 'D123E', 'I135L', 'D177E', 'M184V', 'E194Q', 'G196K', 'T200A'], ['K46Q', 'S48T', 'A62V', 'K65R', 'S68G', 'K70E', 'L74I', 'A98S', 'K101Q', 'K122E', 'I135T', 'S162C', 'I178L', 'M184V', 'V189I', 'G190S', 'T200A', 'R211K']]
